# FraudGuard — IEEE-CIS Kaggle Training

Attach **IEEE-CIS Fraud Detection** through Kaggle's **Add Input**, then run all cells. The notebook trains LightGBM and Gaussian Naive Bayes, uses a chronological split, evaluates fraud-focused metrics and missing-field stress, and saves artifacts to `/kaggle/working/fraudguard_artifacts.zip`.

Dataset is not copied into the output archive. This notebook is a baseline training pipeline; it does not yet implement the proposed Bayesian Network or EM experiment.

In [14]:
# Kaggle typically has LightGBM. Install pgmpy in Kaggle if missing and Internet is enabled.
# %pip -q install pgmpy==0.1.24 lightgbm


import os, json, time, warnings, platform
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, roc_curve
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import sklearn
import pgmpy

SEED = 42
rng = np.random.default_rng(SEED)
KAGGLE_DATA_DIR = Path('/kaggle/input/competitions/ieee-fraud-detection')
LOCAL_DATA_DIR = Path.cwd() / 'ieee-fraud-detection'
DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else LOCAL_DATA_DIR
OUT = Path('/kaggle/working/fraudguard_artifacts') if Path('/kaggle/working').exists() else Path.cwd() / 'artifacts'
OUT.mkdir(parents=True, exist_ok=True)
print('pgmpy:', pgmpy.__version__, '| LightGBM:', lgb.__version__)

Note: you may need to restart the kernel to use updated packages.
pgmpy: 1.1.2 | LightGBM: 4.6.0


In [15]:
# Load and join the labeled training files. Never use test files for model evaluation.
tx_path = DATA_DIR / 'train_transaction.csv'
id_path = DATA_DIR / 'train_identity.csv'
if not tx_path.exists():
    raise FileNotFoundError(f'Cannot find {tx_path}. Attach IEEE-CIS through Kaggle > Add Input.')

tx = pd.read_csv(tx_path, low_memory=False)
if id_path.exists():
    ident = pd.read_csv(id_path, low_memory=False)
    data = tx.merge(ident, on='TransactionID', how='left', validate='one_to_one')
    del ident
else:
    warnings.warn('train_identity.csv missing; proceeding without identity fields.')
    data = tx.copy()
del tx

data = data.sort_values('TransactionDT', kind='mergesort').reset_index(drop=True)
split = int(0.80 * len(data))
train_df = data.iloc[:split].copy()
valid_df = data.iloc[split:].copy()
y_train = train_df['isFraud'].astype('int8')
y_valid = valid_df['isFraud'].astype('int8')
print('Train:', train_df.shape, '| Validation:', valid_df.shape)
print('Train fraud rate:', round(y_train.mean(), 5), '| Validation fraud rate:', round(y_valid.mean(), 5))

Train: (472432, 434) | Validation: (118108, 434)
Train fraud rate: 0.03514 | Validation fraud rate: 0.03441


In [16]:
def recall_at_fpr(y, scores, target_fpr=0.05):
    fpr, tpr, thresholds = roc_curve(y, scores)
    eligible = np.flatnonzero(fpr <= target_fpr)
    i = eligible[np.argmax(tpr[eligible])]
    return float(tpr[i]), float(thresholds[i])

def evaluate(name, y, scores):
    recall, threshold = recall_at_fpr(y, scores, 0.05)
    return {
        'model': name,
        'PR_AUC': float(average_precision_score(y, scores)),
        'ROC_AUC': float(roc_auc_score(y, scores)),
        'Brier': float(brier_score_loss(y, scores)),
        'Recall_at_5pct_FPR': recall,
        'Threshold_at_5pct_FPR': threshold
    }

# Common LightGBM feature preparation; excludes target and ID.
drop_cols = ['isFraud', 'TransactionID']
Xtr0 = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns]).copy()
Xva0 = valid_df.drop(columns=[c for c in drop_cols if c in valid_df.columns]).copy()
cat_candidates = ['ProductCD','card1','card2','card3','card4','card5','card6',
                  'addr1','addr2','P_emaildomain','R_emaildomain','DeviceType','DeviceInfo']
cat_cols = [c for c in cat_candidates if c in Xtr0.columns]
num_cols = [c for c in Xtr0.columns if c not in cat_cols and pd.api.types.is_numeric_dtype(Xtr0[c])]
features = [c for c in num_cols + cat_cols if Xtr0[c].nunique(dropna=True) > 1]
cat_cols = [c for c in cat_cols if c in features]
num_cols = [c for c in num_cols if c in features]
X_train = Xtr0[features].copy()
X_valid = Xva0[features].copy()
category_levels = {}

for c in cat_cols:
    a = X_train[c].astype('string').fillna('__MISSING__').astype(str)
    b = X_valid[c].astype('string').fillna('__MISSING__').astype(str)
    levels = sorted(set(a.unique()) | {'__MISSING__', '__UNKNOWN__'})
    category_levels[c] = levels
    b = b.where(b.isin(set(levels)), '__UNKNOWN__')
    X_train[c] = pd.Categorical(a, categories=levels)
    X_valid[c] = pd.Categorical(b, categories=levels)
for frame in (X_train, X_valid):
    frame.replace([np.inf, -np.inf], np.nan, inplace=True)

lgb_model = lgb.LGBMClassifier(
    objective='binary', n_estimators=1200, learning_rate=0.03, num_leaves=31,
    subsample=0.85, colsample_bytree=0.85, reg_lambda=2.0,
    random_state=SEED, n_jobs=-1, verbosity=-1
)
lgb_model.fit(X_train, y_train, categorical_feature=cat_cols,
              eval_set=[(X_valid, y_valid)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
p_lgb = lgb_model.predict_proba(X_valid)[:, 1]

nb_imputer = SimpleImputer(strategy='median', add_indicator=True)
Xtr_nb = nb_imputer.fit_transform(X_train[num_cols])
Xva_nb = nb_imputer.transform(X_valid[num_cols])
nb_model = GaussianNB().fit(Xtr_nb, y_train)
p_nb = nb_model.predict_proba(Xva_nb)[:, 1]
print('Baselines trained.')

Baselines trained.


In [ ]:
# Prepare compact discrete variables for the Bayesian Network.
# Bin edges and category frequency decisions are learned from training rows only.
def quantile_edges(series, q=5):
    values = pd.to_numeric(series, errors='coerce').dropna()
    if values.empty:
        return np.array([-np.inf, 0.0, np.inf])
    edges = np.unique(values.quantile(np.linspace(0, 1, q + 1)).to_numpy())
    if len(edges) < 3:
        lo, hi = float(values.min()), float(values.max())
        if lo == hi: hi = lo + 1.0
        return np.array([-np.inf, lo, hi, np.inf], dtype=float)
    edges[0], edges[-1] = -np.inf, np.inf
    return edges

bn_train_raw = pd.DataFrame(index=train_df.index)
bn_valid_raw = pd.DataFrame(index=valid_df.index)
bn_bin_edges = {}
for col, q in [('TransactionAmt', 5), ('TransactionDT', 5), ('card1', 5), ('addr1', 4)]:
    if col in train_df.columns:
        edges_col = quantile_edges(train_df[col], q)
        bn_bin_edges[col] = edges_col.tolist()
        bn_train_raw[col + 'Bin'] = pd.cut(pd.to_numeric(train_df[col], errors='coerce'), bins=edges_col, labels=False, include_lowest=True)
        bn_valid_raw[col + 'Bin'] = pd.cut(pd.to_numeric(valid_df[col], errors='coerce'), bins=edges_col, labels=False, include_lowest=True)

bn_category_levels = {}
for col in [c for c in ['ProductCD', 'card4', 'DeviceType', 'P_emaildomain'] if c in train_df.columns]:
    trv = train_df[col].astype('string').fillna('__MISSING__').astype(str)
    vav = valid_df[col].astype('string').fillna('__MISSING__').astype(str)
    freq = trv.value_counts()
    keep = set(freq[freq >= max(100, int(0.001 * len(trv)))].index)
    levels = sorted(keep | {'__OTHER__', '__MISSING__'})
    bn_category_levels[col] = levels
    bn_train_raw[col] = trv.where(trv.isin(keep), '__OTHER__')
    bn_valid_raw[col] = vav.where(vav.isin(keep), '__OTHER__')

bn_train_raw['Fraud'] = y_train.to_numpy()
bn_valid_raw['Fraud'] = y_valid.to_numpy()
bn_feature_candidates = ['TransactionAmtBin','TransactionDTBin','card1Bin','ProductCD','card4','DeviceType','P_emaildomain','addr1Bin']
bn_features = [c for c in bn_feature_candidates if c in bn_train_raw.columns]
bn_fit = bn_train_raw[bn_features + ['Fraud']].dropna().copy()

MAX_BN_ROWS = 150_000
if len(bn_fit) > MAX_BN_ROWS:
    parts = []
    total_bn_rows = len(bn_fit)
    for _, group in bn_fit.groupby('Fraud', sort=False):
        n_take = max(1, int(MAX_BN_ROWS * len(group) / total_bn_rows))
        parts.append(group.sample(n=min(n_take, len(group)), random_state=SEED))
    bn_fit = pd.concat(parts).sample(frac=1, random_state=SEED)
    if len(bn_fit) > MAX_BN_ROWS:
        bn_fit = bn_fit.sample(MAX_BN_ROWS, random_state=SEED)

edges = [(c, 'Fraud') for c in bn_features]
if 'ProductCD' in bn_features and 'addr1Bin' in bn_features:
    edges.append(('ProductCD', 'addr1Bin'))
print('BN features:', bn_features, '| complete cases:', len(bn_fit))


In [17]:
# Version-compatible pgmpy fitting: estimate CPDs explicitly, then attach them.
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

bn_model = BayesianNetwork(edges)
bn_estimator = BayesianEstimator(bn_model, bn_fit)
bn_cpds = bn_estimator.get_parameters(
    prior_type='BDeu',
    equivalent_sample_size=10
)
bn_model.add_cpds(*bn_cpds)
assert bn_model.check_model(), 'Bayesian Network CPDs failed model validation.'
bn_infer = VariableElimination(bn_model)

print('BN features:', bn_features)
print('Complete training rows for BN:', len(bn_fit))
print('BN graph edges:', list(bn_model.edges()))

/tmp/ipykernel_219/1304812333.py:7: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  bn_estimator = BayesianEstimator(bn_model, bn_fit)


BN features: ['TransactionAmtBin', 'TransactionDTBin', 'card1Bin', 'ProductCD', 'card4', 'DeviceType', 'P_emaildomain', 'addr1Bin']
Complete training rows for BN: 149999
BN graph edges: [('TransactionAmtBin', 'Fraud'), ('TransactionDTBin', 'Fraud'), ('card1Bin', 'Fraud'), ('ProductCD', 'Fraud'), ('ProductCD', 'addr1Bin'), ('card4', 'Fraud'), ('DeviceType', 'Fraud'), ('P_emaildomain', 'Fraud'), ('addr1Bin', 'Fraud')]


In [18]:
# Bayesian Network prediction with missing evidence marginalized.
# Exact inference is done on a reproducible validation sample for practical runtime.
# Raise BN_EVAL_ROWS after confirming runtime/memory on Kaggle.
BN_EVAL_ROWS = min(3000, len(bn_valid_raw))
bn_eval_idx = np.linspace(0, len(bn_valid_raw)-1, BN_EVAL_ROWS, dtype=int)
bn_eval = bn_valid_raw.iloc[bn_eval_idx].copy()
bn_y = bn_eval['Fraud'].astype(int).to_numpy()

def bn_score_row(row):
    evidence = {}
    for c in bn_features:
        val = row[c]
        if pd.isna(val):
            continue  # missing evidence is marginalized by variable elimination
        if c in bn_category_levels and val not in bn_category_levels[c]:
            val = '__OTHER__'
        evidence[c] = val
    try:
        q = bn_infer.query(variables=['Fraud'], evidence=evidence, show_progress=False)
        states = list(q.state_names['Fraud'])
        return float(q.values[states.index(1)]) if 1 in states else float(q.values[-1])
    except Exception:
        return float(y_train.mean())

t0 = time.time()
p_bn = np.array([bn_score_row(row) for _, row in bn_eval.iterrows()])
print('BN inference seconds:', round(time.time()-t0, 2), '| evaluated rows:', len(bn_eval))
print('BN metrics:', evaluate('BayesianNetwork', bn_y, p_bn))

BN inference seconds: 2.52 | evaluated rows: 3000
BN metrics: {'model': 'BayesianNetwork', 'PR_AUC': 0.07819574724713363, 'ROC_AUC': 0.7450380161297768, 'Brier': 0.06370544728729452, 'Recall_at_5pct_FPR': 0.010309278350515464, 'Threshold_at_5pct_FPR': 0.9999981481550069}


In [20]:
# Select a small numeric feature set and fit scaling only on training rows.
gmm_candidates = [c for c in ['TransactionAmt', 'TransactionDT', 'card1', 'card2', 'card3', 'addr1'] if c in train_df.columns]
gmm_imputer = SimpleImputer(strategy='median')
gmm_scaler = StandardScaler()

# Use legitimate training transactions only; cap sample for reasonable Kaggle runtime.
legit_mask = (y_train.to_numpy() == 0)
Xg_raw = train_df.loc[legit_mask, gmm_candidates].replace([np.inf, -np.inf], np.nan)
Xg_imp = gmm_imputer.fit_transform(Xg_raw)
Xg_scaled = gmm_scaler.fit_transform(Xg_imp)
if len(Xg_scaled) > 100_000:
    take = rng.choice(len(Xg_scaled), 100_000, replace=False)
    Xg_fit = Xg_scaled[take]
else:
    Xg_fit = Xg_scaled

gmm = GaussianMixture(
    n_components=4, covariance_type='diag', reg_covar=1e-4,
    max_iter=100, n_init=2, random_state=SEED
)
t0 = time.time()
gmm.fit(Xg_fit)  # sklearn GaussianMixture uses EM
print('GMM converged:', gmm.converged_, '| iterations:', gmm.n_iter_, '| seconds:', round(time.time()-t0, 2))

Xg_valid = valid_df[gmm_candidates].replace([np.inf, -np.inf], np.nan)
Xg_valid = gmm_scaler.transform(gmm_imputer.transform(Xg_valid))
gmm_anomaly = -gmm.score_samples(Xg_valid)  # higher = less likely under legitimate-transaction mixture
gmm_metrics = {
    'model': 'GMM_EM_anomaly_score',
    'PR_AUC': float(average_precision_score(y_valid, gmm_anomaly)),
    'ROC_AUC': float(roc_auc_score(y_valid, gmm_anomaly))
}
print('GMM-EM metrics:', gmm_metrics)

GMM converged: True | iterations: 16 | seconds: 1.98
GMM-EM metrics: {'model': 'GMM_EM_anomaly_score', 'PR_AUC': 0.033761968838485784, 'ROC_AUC': 0.41683638534526113}


In [21]:
rows = [
    evaluate('LightGBM', y_valid, p_lgb),
    evaluate('GaussianNB_numeric', y_valid, p_nb),
    {**gmm_metrics, 'Brier': np.nan, 'Recall_at_5pct_FPR': np.nan, 'Threshold_at_5pct_FPR': np.nan},
    {**evaluate('BayesianNetwork_sample', bn_y, p_bn), 'validation_rows_used': len(bn_y)}
]
results = pd.DataFrame(rows)
display(results)
results.to_csv(OUT / 'model_comparison.csv', index=False)

# Re-evaluate LightGBM after masking selected identity/contact fields at 10%, 30%, 60%, and 100%.
mask_cols = [c for c in ['DeviceType','DeviceInfo','P_emaildomain','R_emaildomain','addr1','addr2'] if c in X_valid.columns]
mask_rows = [evaluate('LightGBM', y_valid, p_lgb)]
for frac in [0.10, 0.30, 0.60, 1.00]:
    Xm = X_valid.copy()
    for c in mask_cols:
        vals = Xm[c].astype('string').fillna('__MISSING__').astype(str)
        observed = np.flatnonzero(vals.to_numpy() != '__MISSING__')
        n = int(round(frac * len(observed)))
        if n:
            chosen = rng.choice(observed, size=n, replace=False)
            vals.iloc[chosen] = '__MISSING__'
        Xm[c] = pd.Categorical(vals, categories=category_levels[c])
    pm = lgb_model.predict_proba(Xm)[:, 1]
    mask_rows.append(evaluate('LightGBM', y_valid, pm) | {'mask_fraction_per_field': frac})
mask_df = pd.DataFrame(mask_rows)
display(mask_df)
mask_df.to_csv(OUT / 'missingness_stress.csv', index=False)

print('Masked fields:', mask_cols)

,model,PR_AUC,ROC_AUC,Brier,Recall_at_5pct_FPR,Threshold_at_5pct_FPR,validation_rows_used
0,LightGBM,0.559487,0.910000,0.020971,0.657972,0.066021,NaN
1,GaussianNB_numeric,0.088913,0.734347,0.695684,0.178396,0.999968,NaN
2,GMM_EM_anomaly_score,0.033762,0.416836,NaN,NaN,NaN,NaN
3,BayesianNetwork_sample,0.078196,0.745038,0.063705,0.010309,0.999998,3000.0


,model,PR_AUC,ROC_AUC,Brier,Recall_at_5pct_FPR,Threshold_at_5pct_FPR,mask_fraction_per_field
0,LightGBM,0.559487,0.910000,0.020971,0.657972,0.066021,NaN
1,LightGBM,0.555744,0.908892,0.021093,0.652067,0.068348,0.1
2,LightGBM,0.547216,0.906468,0.021357,0.647884,0.072980,0.3
3,LightGBM,0.535867,0.901710,0.021737,0.636319,0.082542,0.6
4,LightGBM,0.520108,0.895323,0.022278,0.618602,0.093866,1.0


Masked fields: ['DeviceType', 'DeviceInfo', 'P_emaildomain', 'R_emaildomain', 'addr1', 'addr2']


In [22]:
# Save models, preprocessing objects, experiment results, and metadata.
joblib.dump(lgb_model, OUT / 'lightgbm_model.joblib', compress=3)
joblib.dump(nb_model, OUT / 'gaussian_nb_model.joblib', compress=3)
joblib.dump(nb_imputer, OUT / 'nb_imputer.joblib', compress=3)
joblib.dump(bn_model, OUT / 'bayesian_network.joblib', compress=3)
joblib.dump(gmm, OUT / 'gmm_em_model.joblib', compress=3)
joblib.dump(gmm_imputer, OUT / 'gmm_imputer.joblib', compress=3)
joblib.dump(gmm_scaler, OUT / 'gmm_scaler.joblib', compress=3)

metadata = {
    'project': 'FraudGuard',
    'dataset': 'IEEE-CIS Fraud Detection',
    'split': 'chronological 80/20 by TransactionDT',
    'bn_features': bn_features,
    'bn_bin_edges': bn_bin_edges,
    'bn_edges': list(map(list, bn_model.edges())),
    'bn_training_complete_case_rows': int(len(bn_fit)),
    'bn_validation_sample_rows': int(len(bn_y)),
    'gmm_features': gmm_candidates,
    'gmm_components': 4,
    'gmm_converged': bool(gmm.converged_),
    'gmm_iterations': int(gmm.n_iter_),
    'seed': SEED,
    'note': 'GMM uses EM; BN missing evidence is marginalized at inference. BN CPDs fit on complete cases.'
}
(OUT / 'metadata.json').write_text(json.dumps(metadata, indent=2))
zip_path = (Path('/kaggle/working/fraudguard_artifacts.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'fraudguard_artifacts.zip')
import zipfile
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        if f.is_file():
            z.write(f, arcname=f'fraudguard_artifacts/{f.name}')
print('Saved:', zip_path)
for f in sorted(OUT.iterdir()):
    if f.is_file(): print(f.name, round(f.stat().st_size/1024/1024, 2), 'MB')

Saved: /kaggle/working/fraudguard_artifacts.zip
bayesian_network.joblib 0.21 MB
gaussian_nb_model.joblib 0.01 MB
gmm_em_model.joblib 0.0 MB
gmm_imputer.joblib 0.0 MB
gmm_scaler.joblib 0.0 MB
missingness_stress.csv 0.0 MB
lightgbm_model.joblib 1.65 MB
metadata.json 0.0 MB
nb_imputer.joblib 0.0 MB
model_comparison.csv 0.0 MB
